In [ ]:
# Example 1: protocol constants + method registries from acp.meta
from acp import AGENT_METHODS, CLIENT_METHODS, PROTOCOL_VERSION

print(f"PROTOCOL_VERSION = {PROTOCOL_VERSION}")
print(f"AGENT_METHODS has {len(AGENT_METHODS)} entries")
print(f"CLIENT_METHODS has {len(CLIENT_METHODS)} entries")

# Sample a few well-known JSON-RPC method names used by ACP.
agent_method_names = set(AGENT_METHODS.values()) if isinstance(AGENT_METHODS, dict) else set(AGENT_METHODS)
print("sample agent methods:", sorted(list(agent_method_names))[:5])

assert PROTOCOL_VERSION >= 1
assert AGENT_METHODS and CLIENT_METHODS

In [ ]:
# Example 2: build content blocks via the helper factories in acp.helpers
from acp import (
    audio_block,
    embedded_text_resource,
    image_block,
    resource_link_block,
    text_block,
)

txt = text_block("hello, agent")
img = image_block(data="aGVsbG8=", mime_type="image/png")
aud = audio_block(data="aGVsbG8=", mime_type="audio/wav")
link = resource_link_block(uri="file:///tmp/x.txt", name="x.txt")
emb = embedded_text_resource(uri="file:///tmp/x.txt", text="file body")

for block in (txt, img, aud, link, emb):
    print(type(block).__name__, "->", block.model_dump(exclude_none=True))

assert txt.text == "hello, agent"
assert img.mime_type == "image/png"

In [ ]:
# Example 3: assemble + validate JSON-RPC request/response models from acp.schema
from acp import (
    InitializeRequest,
    InitializeResponse,
    NewSessionRequest,
    PromptRequest,
    PROTOCOL_VERSION,
)
from acp import text_block

init_req = InitializeRequest(protocol_version=PROTOCOL_VERSION, client_capabilities={})
init_resp = InitializeResponse(protocol_version=PROTOCOL_VERSION, agent_capabilities={}, auth_methods=[])
new_sess = NewSessionRequest(cwd="/tmp", mcp_servers=[])
prompt = PromptRequest(session_id="sess-1", prompt=[text_block("summarize this")])

for model in (init_req, init_resp, new_sess, prompt):
    payload = model.model_dump_json(exclude_none=True)
    print(type(model).__name__, len(payload), "bytes")

assert init_req.protocol_version == PROTOCOL_VERSION
assert prompt.session_id == "sess-1"

In [ ]:
# Example 4: SessionNotification + update helpers (agent->client streaming chunks)
from acp import (
    plan_entry,
    session_notification,
    update_agent_message_text,
    update_agent_thought_text,
    update_plan,
)
from acp import SessionNotification

msg_update = update_agent_message_text("working on it")
thought_update = update_agent_thought_text("considering options")
plan_update = update_plan([
    plan_entry(content="step 1", priority="high", status="pending"),
    plan_entry(content="step 2", priority="medium", status="pending"),
])

for update in (msg_update, thought_update, plan_update):
    note = session_notification(session_id="sess-1", update=update)
    assert isinstance(note, SessionNotification)
    print(type(update).__name__, "->", note.session_id)

print("plan entries:", len(plan_update.entries))

In [ ]:
# Example 5: top-level public symbols include Agent/Client + connect_to_agent
import inspect

import acp
from acp import Agent, Client, RequestError, connect_to_agent, run_agent

# Sanity-check the connection entry points exist with the right shape (don't call them: needs a real subprocess).
assert inspect.isclass(Agent)
assert inspect.isclass(Client)
assert issubclass(RequestError, Exception)
assert callable(connect_to_agent)
assert callable(run_agent)

# Verify Agent/Client expose at least one ACP method handler hook.
agent_members = {n for n, _ in inspect.getmembers(Agent)}
client_members = {n for n, _ in inspect.getmembers(Client)}
print("Agent has", len(agent_members), "members; Client has", len(client_members))

print("acp top-level symbol count:", sum(1 for _ in dir(acp) if not _.startswith("_")))
assert "PROTOCOL_VERSION" in dir(acp)